# Link to Colab:
https://colab.research.google.com/drive/1xw0unwQP6ZRyLIiM6_yvoNOBGbt4hFfd?usp=sharing

# Set-Up

In [8]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Create and move into a specific project folder
import os
project_folder = '/content/drive/MyDrive/Colab_Projects/topic1'

if not os.path.exists(project_folder):
    os.makedirs(project_folder)
    print(f"Created folder: {project_folder}")

# 3. Change the working directory to this folder
os.chdir(project_folder)
print(f"Current Directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Created folder: /content/drive/MyDrive/Colab_Projects/topic1
Current Directory: /content/drive/MyDrive/Colab_Projects/topic1


In [9]:
# Run this in a cell
!nvidia-smi

Sun Mar  1 23:56:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   28C    P0             40W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [10]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && \
sudo apt-get install -y nodejs && \
sudo npm install -g @anthropic-ai/claude-code && \
export PATH=/usr/bin:$PATH

2026-03-01 23:56:24 - Installing pre-requisites
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://deb.nodesource.com/node_20.x nodistro InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
85 packages can be upgrad

In [11]:
# Install required packages
!pip install -q transformers torch datasets accelerate tqdm huggingface_hub bitsandbytes

In [12]:
from google.colab import userdata

# Login to Hugging Face
hf_token = userdata.get('HF_TOKEN')
!huggingface-cli login --token $hf_token

/bin/bash: line 1: huggingface-cli: command not found


# Create Multi-Model MMLU Evaluation Script
This script evaluates multiple small language models on the MMLU benchmark:
- Llama 3.2-1B-Instruct
- Qwen 2.5 0.5B-Instruct
- OLMo 2 1B

As well as these additional small/medium language models:
- Llama 3.2 3B
- Qwen 2.5 7B
- OLMo 3 Base 7B

In [13]:
%%writefile multimodel_mmlu_eval.py

"""
Multi-Model MMLU Evaluation Script (Optimized for Google Colab A100)

This script evaluates multiple small/medium language models on the MMLU benchmark:
- Llama 3.2 1B-Instruct
- Qwen 2.5 0.5B-Instruct
- OLMo 2 1B
- Llama 3.2 3B-Instruct
- Qwen 2.5 7B-Instruct
- OLMo 3 Base 7B

Optimized for A100 GPU:
- bfloat16 precision (A100 has native BF16 tensor cores — faster & more stable than FP16)
- No quantization needed (A100 40GB easily fits all models: ~2 GB for 1B, ~14 GB for 7B)
- torch.compile() for additional inference speedup
- Models are evaluated sequentially and unloaded between runs to keep memory clean

Usage:
1. Open in Google Colab with A100 runtime (Runtime > Change runtime type > A100)
2. Install: pip install transformers torch datasets accelerate tqdm
3. Login: huggingface-cli login
4. Run: python multimodel_mmlu_eval.py

Set PRINT_QUESTIONS to True to see each question and answer.
Set USE_TORCH_COMPILE to False if you hit any compilation errors.
"""

import argparse
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import json
from tqdm.auto import tqdm
from datetime import datetime
import sys
import platform
import time
import gc

# ============================================================================
# CONFIGURATION - Modify these settings
# ============================================================================

# Models to evaluate (ordered small to large)
MODELS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "allenai/OLMo-1B-0724-hf",          # OLMo 2 1B
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "allenai/OLMo-3-7B"                  # OLMo 3 Base 7B
]

# A100 precision settings
# bfloat16 is the recommended dtype for A100:
#   - Native BF16 tensor cores -> faster than FP16
#   - Wider dynamic range than FP16 -> more numerically stable
#   - No quantization needed; A100 (40 GB) fits all models at full BF16 precision
TORCH_DTYPE = torch.bfloat16

# torch.compile() fuses ops into optimized CUDA kernels, giving a meaningful
# speedup on A100. Adds ~1-2 min compilation overhead the first time each model
# runs. Disable with --no-compile or by setting False below.
USE_TORCH_COMPILE = True

MAX_NEW_TOKENS = 1

# Print detailed question/answer information
PRINT_QUESTIONS = False  # Set to True to see each question, answer, and result

# 10 MMLU subjects to evaluate
MMLU_SUBJECTS = [
    "astronomy",
    "college_computer_science",
    "elementary_mathematics",
    "human_sexuality",
    "international_law",
    "logical_fallacies",
    "moral_scenarios",
    "philosophy",
    "professional_medicine",
    "professional_psychology"
]


# ============================================================================
# ENVIRONMENT CHECK
# ============================================================================

def check_environment():
    """Verify we are running on an A100 and that dependencies are in order."""
    print("=" * 70)
    print("Environment Check")
    print("=" * 70)

    # Colab detection
    try:
        import google.colab
        print("✓ Running in Google Colab")
        in_colab = True
    except ImportError:
        print("✓ Running locally (not in Colab)")
        in_colab = False

    print(f"✓ Platform: {platform.system()} ({platform.machine()})")

    # GPU check
    if not torch.cuda.is_available():
        print("❌ No CUDA GPU detected.")
        print("   This script is optimized for an A100. Please switch your Colab")
        print("   runtime: Runtime > Change runtime type > GPU > A100.")
        sys.exit(1)

    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {gpu_name}")
    print(f"✓ VRAM: {gpu_memory_gb:.1f} GB")

    if "A100" not in gpu_name:
        print(f"⚠️  Expected an A100 but found '{gpu_name}'.")
        print("   The script will still run, but settings are tuned for A100.")

    # BF16 support check
    global TORCH_DTYPE
    if torch.cuda.is_bf16_supported():
        print("✓ bfloat16 supported — using native A100 BF16 tensor cores")
    else:
        print("⚠️  bfloat16 not supported on this GPU. Falling back to float16.")
        TORCH_DTYPE = torch.float16

    # torch.compile check
    global USE_TORCH_COMPILE
    if USE_TORCH_COMPILE:
        if int(torch.__version__.split(".")[0]) >= 2:
            print(f"✓ torch.compile() available (PyTorch {torch.__version__})")
        else:
            print(f"⚠️  torch.compile() requires PyTorch >= 2.0 (found {torch.__version__}). Disabling.")
            USE_TORCH_COMPILE = False

    # HF authentication
    try:
        from huggingface_hub import HfFolder
        if HfFolder.get_token():
            print("✓ Hugging Face authenticated")
        else:
            print("⚠️  No Hugging Face token — run: huggingface-cli login")
    except Exception:
        print("⚠️  Could not check Hugging Face authentication")

    # Configuration summary
    print("\n" + "=" * 70)
    print("Configuration")
    print("=" * 70)
    print(f"Models to evaluate : {len(MODELS)}")
    for m in MODELS:
        print(f"  - {m}")
    print(f"Precision          : {TORCH_DTYPE}  (BF16 — optimal for A100)")
    print(f"Quantization       : None  (not needed with {gpu_memory_gb:.0f} GB VRAM)")
    print(f"torch.compile()    : {USE_TORCH_COMPILE}")
    print(f"Subjects           : {len(MMLU_SUBJECTS)}")
    print(f"Print questions    : {PRINT_QUESTIONS}")
    print()
    print("Approximate BF16 VRAM per model:")
    print("  0.5B -> ~1 GB  |  1B -> ~2 GB  |  3B -> ~6 GB  |  7B -> ~14 GB")
    print("=" * 70 + "\n")

    return in_colab


# ============================================================================
# MODEL LOADING / UNLOADING
# ============================================================================

def load_model_and_tokenizer(model_name: str):
    """Load a model in BF16 on CUDA with optional torch.compile()."""
    print(f"\nLoading {model_name} ...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("  ✓ Tokenizer loaded")

    print("  Loading weights in BF16 (may take 1-3 min for 7B models)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=TORCH_DTYPE,    # BF16 — native A100 tensor-core format
        device_map="auto",          # Automatically places all layers on GPU
        low_cpu_mem_usage=True      # Reduces peak CPU RAM during loading
    )
    model.eval()

    if USE_TORCH_COMPILE:
        print("  Compiling model with torch.compile() ...")
        try:
            model = torch.compile(model)
            print("  ✓ torch.compile() applied")
        except Exception as e:
            print(f"  ⚠️  torch.compile() failed ({e}). Continuing without compilation.")

    mem_alloc = torch.cuda.memory_allocated(0) / 1e9
    mem_reserv = torch.cuda.memory_reserved(0) / 1e9
    print(f"  ✓ Ready  |  dtype: {TORCH_DTYPE}  |  "
          f"VRAM: {mem_alloc:.2f} GB allocated, {mem_reserv:.2f} GB reserved")

    return model, tokenizer


def unload_model(model, tokenizer):
    """Delete model and free GPU memory before loading the next one."""
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    mem = torch.cuda.memory_allocated(0) / 1e9
    print(f"  ✓ Model unloaded  |  VRAM now: {mem:.2f} GB allocated")


# ============================================================================
# INFERENCE UTILITIES
# ============================================================================

def format_mmlu_prompt(question: str, choices: list) -> str:
    """Format an MMLU question as a standard multiple-choice prompt."""
    prompt = f"{question}\n\n"
    for label, choice in zip(("A", "B", "C", "D"), choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer:"
    return prompt


class TimingInfo:
    """Accumulate wall-clock, CPU, and GPU timing across inference calls."""

    def __init__(self):
        self.real_time = 0.0
        self.cpu_time = 0.0
        self.gpu_time = 0.0
        self.num_questions = 0

    def start(self):
        self._t_real = time.time()
        self._t_cpu = time.process_time()
        torch.cuda.synchronize()
        self._ev_start = torch.cuda.Event(enable_timing=True)
        self._ev_end = torch.cuda.Event(enable_timing=True)
        self._ev_start.record()

    def stop(self):
        self._ev_end.record()
        torch.cuda.synchronize()
        self.gpu_time += self._ev_start.elapsed_time(self._ev_end) / 1000.0  # ms -> s
        self.real_time += time.time() - self._t_real
        self.cpu_time += time.process_time() - self._t_cpu
        self.num_questions += 1

    def summary(self) -> dict:
        n = max(self.num_questions, 1)
        return {
            "total_real_time_seconds": self.real_time,
            "total_cpu_time_seconds": self.cpu_time,
            "total_gpu_time_seconds": self.gpu_time,
            "avg_real_time_per_question": self.real_time / n,
            "avg_cpu_time_per_question": self.cpu_time / n,
            "avg_gpu_time_per_question": self.gpu_time / n,
            "num_questions": self.num_questions,
        }


def get_model_prediction(model, tokenizer, prompt: str, timing: TimingInfo) -> str:
    """Run one greedy forward pass and return the predicted answer letter."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    timing.start()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            temperature=1.0,
        )
    timing.stop()

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    # Extract first valid answer letter; fall back to "A" if none found
    answer = generated.strip()[:1].upper()
    if answer not in ("A", "B", "C", "D"):
        for ch in generated.upper():
            if ch in ("A", "B", "C", "D"):
                answer = ch
                break
        else:
            answer = "A"

    return answer


# ============================================================================
# EVALUATION
# ============================================================================

def evaluate_subject(model, tokenizer, subject: str, model_name: str):
    """Evaluate a model on one MMLU subject and return a results dict."""
    print(f"\n{'='*70}")
    print(f"Subject : {subject}")
    print(f"Model   : {model_name}")
    print(f"{'='*70}")

    try:
        dataset = load_dataset("cais/mmlu", subject, split="test")
    except Exception as e:
        print(f"❌ Could not load subject '{subject}': {e}")
        return None

    correct = 0
    timing = TimingInfo()
    question_results = []

    for idx, example in enumerate(tqdm(dataset, desc=subject, leave=True)):
        choices = example["choices"]
        correct_answer = ("A", "B", "C", "D")[example["answer"]]
        prompt = format_mmlu_prompt(example["question"], choices)
        predicted = get_model_prediction(model, tokenizer, prompt, timing)

        is_correct = predicted == correct_answer
        if is_correct:
            correct += 1

        question_results.append({
            "question_id": idx,
            "question": example["question"],
            "choices": choices,
            "correct_answer": correct_answer,
            "predicted_answer": predicted,
            "is_correct": is_correct,
        })

        if PRINT_QUESTIONS:
            print(f"\n--- Q{idx + 1} ---")
            print(f"  {example['question']}")
            for i, ch in enumerate(choices):
                lbl = ("A", "B", "C", "D")[i]
                marker = "  <--" if lbl == correct_answer else ""
                print(f"  {lbl}. {ch}{marker}")
            print(f"  Predicted: {predicted}  |  Correct: {correct_answer}  |  "
                  f"{'✓ CORRECT' if is_correct else '✗ WRONG'}")

    total = len(question_results)
    accuracy = (correct / total * 100) if total > 0 else 0.0
    t = timing.summary()

    print(f"  ✓ {correct}/{total} = {accuracy:.2f}%  "
          f"| wall: {t['total_real_time_seconds']:.1f}s  "
          f"| GPU: {t['total_gpu_time_seconds']:.1f}s  "
          f"| avg: {t['avg_gpu_time_per_question']*1000:.1f}ms/q")

    return {
        "subject": subject,
        "correct": correct,
        "total": total,
        "accuracy": accuracy,
        "timing": t,
        "question_results": question_results,
    }


def evaluate_model(model_name: str) -> dict:
    """Load, evaluate, and unload one model across all MMLU subjects."""
    print(f"\n{'#'*70}")
    print(f"# EVALUATING: {model_name}")
    print(f"{'#'*70}\n")

    model, tokenizer = load_model_and_tokenizer(model_name)

    results = []
    total_correct = 0
    total_questions = 0
    agg = TimingInfo()

    wall_start = time.time()
    cpu_start = time.process_time()

    for i, subject in enumerate(MMLU_SUBJECTS, 1):
        print(f"\nSubject {i}/{len(MMLU_SUBJECTS)}")
        result = evaluate_subject(model, tokenizer, subject, model_name)
        if result:
            results.append(result)
            total_correct += result["correct"]
            total_questions += result["total"]
            t = result["timing"]
            agg.real_time += t["total_real_time_seconds"]
            agg.cpu_time += t["total_cpu_time_seconds"]
            agg.gpu_time += t["total_gpu_time_seconds"]
            agg.num_questions += t["num_questions"]

    wall_elapsed = time.time() - wall_start
    cpu_elapsed = time.process_time() - cpu_start
    overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0.0

    unload_model(model, tokenizer)

    return {
        "model": model_name,
        "overall_accuracy": overall_accuracy,
        "total_correct": total_correct,
        "total_questions": total_questions,
        "subject_results": results,
        "timing": agg.summary(),
        "wall_clock_time": wall_elapsed,
        "total_cpu_time": cpu_elapsed,
    }


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("\n" + "=" * 70)
    print("Multi-Model MMLU Evaluation  —  Google Colab A100")
    print("=" * 70 + "\n")

    in_colab = check_environment()
    all_results = []

    for model_name in MODELS:
        try:
            all_results.append(evaluate_model(model_name))
        except Exception as e:
            print(f"\n❌ Error evaluating {model_name}: {e}")
            import traceback
            traceback.print_exc()

    # ── Comparative summary ───────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("COMPARATIVE EVALUATION SUMMARY")
    print("=" * 70)

    for r in all_results:
        print(f"\n{r['model']}")
        print("-" * 70)
        print(f"  Accuracy        : {r['overall_accuracy']:.2f}%  "
              f"({r['total_correct']}/{r['total_questions']})")
        print(f"  Wall-clock time : {r['wall_clock_time']:.1f}s  "
              f"({r['wall_clock_time']/60:.2f} min)")
        t = r["timing"]
        print(f"  GPU time (inf.) : {t['total_gpu_time_seconds']:.1f}s")
        print(f"  Avg / question  : {t['avg_real_time_per_question']*1000:.1f}ms wall  "
              f"| {t['avg_gpu_time_per_question']*1000:.1f}ms GPU")

    print("\n" + "=" * 70)
    print("ACCURACY RANKING")
    print("=" * 70)
    for i, r in enumerate(
        sorted(all_results, key=lambda x: x["overall_accuracy"], reverse=True), 1
    ):
        print(f"  {i}. {r['model']:<47} {r['overall_accuracy']:.2f}%")

    print("\n" + "=" * 70)
    print("SPEED RANKING  (avg GPU time per question)")
    print("=" * 70)
    for i, r in enumerate(
        sorted(all_results, key=lambda x: x["timing"]["avg_gpu_time_per_question"]), 1
    ):
        ms = r["timing"]["avg_gpu_time_per_question"] * 1000
        print(f"  {i}. {r['model']:<47} {ms:.1f}ms / question")

    # ── Save results ──────────────────────────────────────────────────────────
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"multimodel_mmlu_results_a100_{timestamp}.json"

    with open(output_file, "w") as f:
        json.dump(
            {
                "timestamp": timestamp,
                "device": torch.cuda.get_device_name(0),
                "torch_dtype": str(TORCH_DTYPE),
                "torch_compile": USE_TORCH_COMPILE,
                "quantization_bits": None,
                "subjects": MMLU_SUBJECTS,
                "num_subjects": len(MMLU_SUBJECTS),
                "results": all_results,
            },
            f,
            indent=2,
        )
    print(f"\n✓ Results saved -> {output_file}")

    # ── Subject breakdown for the best model ──────────────────────────────────
    if all_results:
        best = max(all_results, key=lambda x: x["overall_accuracy"])
        print(f"\n📊 Subject breakdown — best model: {best['model']}")
        ranked = sorted(best["subject_results"], key=lambda x: x["accuracy"], reverse=True)
        print("  Top 5 subjects:")
        for i, s in enumerate(ranked[:5], 1):
            print(f"    {i}. {s['subject']:<35} {s['accuracy']:.2f}%")
        print("  Bottom 5 subjects:")
        for i, s in enumerate(ranked[-5:], 1):
            print(f"    {i}. {s['subject']:<35} {s['accuracy']:.2f}%")

    if in_colab:
        print("\n" + "=" * 70)
        print("💾 Download results from Colab:")
        print("=" * 70)
        print("  from google.colab import files")
        print(f"  files.download('{output_file}')")

    print("\n✅ Evaluation complete!")
    return output_file


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Multi-model MMLU eval — A100 optimized")
    parser.add_argument("--no-compile", dest="compile", action="store_false",
                        help="Disable torch.compile() (faster startup, slightly slower inference)")
    parser.add_argument("--print-questions", action="store_true",
                        help="Print each question, model prediction, and result")
    args = parser.parse_args()

    USE_TORCH_COMPILE = args.compile
    PRINT_QUESTIONS = args.print_questions

    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
    except Exception as e:
        print(f"\n❌ Fatal error: {e}")
        import traceback
        traceback.print_exc()

Writing multimodel_mmlu_eval.py


# Run Multi-Model MMLU Evaluation Script

In [16]:
!python multimodel_mmlu_eval.py


Multi-Model MMLU Evaluation  —  Google Colab A100

Environment Check
✓ Running in Google Colab
✓ Platform: Linux (x86_64)
✓ GPU: NVIDIA A100-SXM4-40GB
✓ VRAM: 42.4 GB
✓ bfloat16 supported — using native A100 BF16 tensor cores
✓ torch.compile() available (PyTorch 2.10.0+cu128)
⚠️  Could not check Hugging Face authentication

Configuration
Models to evaluate : 6
  - Qwen/Qwen2.5-0.5B-Instruct
  - meta-llama/Llama-3.2-1B-Instruct
  - allenai/OLMo-1B-0724-hf
  - meta-llama/Llama-3.2-3B-Instruct
  - Qwen/Qwen2.5-7B-Instruct
  - allenai/OLMo-3-7B
Precision          : torch.bfloat16  (BF16 — optimal for A100)
Quantization       : None  (not needed with 42 GB VRAM)
torch.compile()    : True
Subjects           : 10
Print questions    : False

Approximate BF16 VRAM per model:
  0.5B -> ~1 GB  |  1B -> ~2 GB  |  3B -> ~6 GB  |  7B -> ~14 GB


######################################################################
# EVALUATING: Qwen/Qwen2.5-0.5B-Instruct
###########################################

In [18]:
from google.colab import files
files.download('multimodel_mmlu_results_a100_20260302_000534.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Create Analysis Script to Visualize Multimodel MMLU Results

In [19]:
%%writefile analyze_mmlu_results.py
"""
Analysis script for multi-model MMLU results.
Creates visualizations and analyzes patterns in model mistakes.

Usage:
    python analyze_mmlu_results.py <results_file.json>
"""

import sys
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================================================
# Load results
# ============================================================================

if len(sys.argv) < 2:
    print("Usage: python analyze_mmlu_results.py <results_file.json>")
    sys.exit(1)

results_file = sys.argv[1]
with open(results_file, 'r') as f:
    data = json.load(f)

models = [r['model'].split('/')[-1] for r in data['results']]
subjects = data['subjects']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#F7DC6F', '#A569BD', '#58D68D']  # supports up to 6 models

has_timing = 'timing' in data['results'][0]
has_questions = (
    data['results'] and
    data['results'][0].get('subject_results') and
    'question_results' in data['results'][0]['subject_results'][0]
)

Path("mmlu_analysis_plots").mkdir(exist_ok=True)

# ============================================================================
# 1. Overall Accuracy Comparison
# ============================================================================
print("Creating accuracy comparison graph...")

overall_accuracies = [r['overall_accuracy'] for r in data['results']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, overall_accuracies, color=colors[:len(models)])
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_title('Overall MMLU Accuracy by Model', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('mmlu_analysis_plots/1_overall_accuracy.png', dpi=300, bbox_inches='tight')
print("  Saved: mmlu_analysis_plots/1_overall_accuracy.png")

# ============================================================================
# 2. Subject-by-Subject Comparison
# ============================================================================
print("Creating subject comparison graph...")

subject_data = {}
for subject in subjects:
    subject_data[subject] = []
    for model_result in data['results']:
        for subj_result in model_result['subject_results']:
            if subj_result['subject'] == subject:
                subject_data[subject].append(subj_result['accuracy'])
                break

fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(len(subjects))
width = 0.13  # narrower to fit 6 models without overlap

for i, model in enumerate(models):
    accuracies = [subject_data[subj][i] for subj in subjects]
    offset = width * (i - (len(models) - 1) / 2)  # dynamically center bars
    ax.bar(x + offset, accuracies, width, label=model, color=colors[i], alpha=0.8)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_xlabel('Subject', fontsize=12)
ax.set_title('Model Performance by Subject', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(subjects, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('mmlu_analysis_plots/2_subject_comparison.png', dpi=300, bbox_inches='tight')
print("  Saved: mmlu_analysis_plots/2_subject_comparison.png")

# ============================================================================
# 3. Timing Comparison (skip if no timing data)
# ============================================================================
if has_timing:
    print("Creating timing comparison graphs...")

    avg_real_times = [r['timing']['avg_real_time_per_question'] for r in data['results']]
    avg_cpu_times  = [r['timing']['avg_cpu_time_per_question']  for r in data['results']]
    avg_gpu_times  = [r['timing'].get('avg_gpu_time_per_question', 0) for r in data['results']]
    has_gpu_time   = any(t > 0 for t in avg_gpu_times)

    ncols = 3 if has_gpu_time else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))

    def _bar(ax, values, title, fmt='.3f'):
        b = ax.bar(models, values, color=colors[:len(models)])
        ax.set_ylabel('Time (seconds)', fontsize=12)
        ax.set_xlabel('Model', fontsize=12)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        for bar in b:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., h,
                    f'{h:{fmt}}s', ha='center', va='bottom', fontsize=9)
        ax.tick_params(axis='x', rotation=45)

    _bar(axes[0], avg_real_times, 'Avg Real Time per Question')
    _bar(axes[1], avg_cpu_times,  'Avg CPU Time per Question')
    if has_gpu_time:
        _bar(axes[2], avg_gpu_times, 'Avg GPU Time per Question')

    plt.tight_layout()
    plt.savefig('mmlu_analysis_plots/3_timing_comparison.png', dpi=300, bbox_inches='tight')
    print("  Saved: mmlu_analysis_plots/3_timing_comparison.png")

    # ============================================================================
    # 4. Accuracy vs Speed Tradeoff
    # ============================================================================
    print("Creating accuracy vs speed graph...")

    fig, ax = plt.subplots(figsize=(10, 8))

    for i, (model, acc, t) in enumerate(zip(models, overall_accuracies, avg_real_times)):
        ax.scatter(t, acc, s=500, alpha=0.6, color=colors[i],
                   edgecolors='black', linewidth=2, label=model)
        ax.annotate(model, (t, acc), xytext=(10, 10),
                    textcoords='offset points', fontsize=10, fontweight='bold')

    ax.set_xlabel('Average Time per Question (seconds)', fontsize=12)
    ax.set_ylabel('Overall Accuracy (%)', fontsize=12)
    ax.set_title('Accuracy vs Speed Tradeoff', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')

    plt.tight_layout()
    plt.savefig('mmlu_analysis_plots/4_accuracy_vs_speed.png', dpi=300, bbox_inches='tight')
    print("  Saved: mmlu_analysis_plots/4_accuracy_vs_speed.png")

# ============================================================================
# 5. Heatmap of Subject Performance
# ============================================================================
print("Creating performance heatmap...")

acc_matrix = np.zeros((len(models), len(subjects)))
for i, model_result in enumerate(data['results']):
    for j, subject in enumerate(subjects):
        for subj_result in model_result['subject_results']:
            if subj_result['subject'] == subject:
                acc_matrix[i, j] = subj_result['accuracy']
                break

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(acc_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(np.arange(len(subjects)))
ax.set_yticks(np.arange(len(models)))
ax.set_xticklabels(subjects, rotation=45, ha='right')
ax.set_yticklabels(models)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Accuracy (%)', rotation=270, labelpad=20)

for i in range(len(models)):
    for j in range(len(subjects)):
        ax.text(j, i, f'{acc_matrix[i, j]:.1f}',
                ha="center", va="center", color="black", fontsize=8)

ax.set_title('Subject Performance Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('mmlu_analysis_plots/5_performance_heatmap.png', dpi=300, bbox_inches='tight')
print("  Saved: mmlu_analysis_plots/5_performance_heatmap.png")

# ============================================================================
# 6. Correct/Wrong Breakdown (pie charts)
# ============================================================================
print("Creating correct/wrong breakdown...")

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 5))
if len(models) == 1:
    axes = [axes]

for i, (model_result, model, ax) in enumerate(zip(data['results'], models, axes)):
    correct = model_result['total_correct']
    wrong   = model_result['total_questions'] - correct
    wedges, texts, autotexts = ax.pie(
        [correct, wrong], labels=['Correct', 'Wrong'],
        autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=90)
    for at in autotexts:
        at.set_color('white')
        at.set_fontsize(11)
        at.set_fontweight('bold')
    ax.set_title(f'{model}\n{correct}/{model_result["total_questions"]} correct',
                 fontsize=11, fontweight='bold')

plt.suptitle('Correct vs Wrong Answers by Model', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('mmlu_analysis_plots/6_correct_wrong_breakdown.png', dpi=300, bbox_inches='tight')
print("  Saved: mmlu_analysis_plots/6_correct_wrong_breakdown.png")

# ============================================================================
# 7. Performance Rankings by Subject
# ============================================================================
print("Creating subject rankings...")

rankings = {subject: [] for subject in subjects}
for subject in subjects:
    subject_accs = subject_data[subject]
    sorted_indices = np.argsort(subject_accs)[::-1]
    for rank, idx in enumerate(sorted_indices):
        rankings[subject].append((models[idx], subject_accs[idx], rank + 1))

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(subjects))
width = 0.13  # narrower to fit 6 models without overlap

for i, model in enumerate(models):
    ranks = []
    for subject in subjects:
        for model_name, acc, rank in rankings[subject]:
            if model_name == model:
                ranks.append(rank)
                break
    offset = width * (i - (len(models) - 1) / 2)  # dynamically center bars
    bars = ax.barh(y_pos + offset, ranks, width, label=model, color=colors[i], alpha=0.8)
    for j, (bar, rank) in enumerate(zip(bars, ranks)):
        ax.text(rank + 0.1, bar.get_y() + bar.get_height()/2,
                f'{rank}', va='center', fontsize=8, fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(subjects)
ax.set_xlabel('Rank (1 = Best)', fontsize=12)
ax.set_title('Model Rankings by Subject', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, len(models) + 1)  # scales to however many models are present
ax.invert_xaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('mmlu_analysis_plots/7_subject_rankings.png', dpi=300, bbox_inches='tight')
print("  Saved: mmlu_analysis_plots/7_subject_rankings.png")

# ============================================================================
# Question-overlap plots (only when question_results are available)
# ============================================================================
if has_questions:

    # Build per-subject overlap counts
    overlap_data = {}
    for subject in subjects:
        subject_results_by_model = []
        for model_result in data['results']:
            for subj_result in model_result['subject_results']:
                if subj_result['subject'] == subject:
                    subject_results_by_model.append(subj_result)
                    break
        if len(subject_results_by_model) < len(data['results']):
            continue

        num_q = subject_results_by_model[0]['total']
        all_correct = all_wrong = mixed = 0
        for q_idx in range(num_q):
            results = [sr['question_results'][q_idx]['is_correct']
                       for sr in subject_results_by_model]
            if all(results):
                all_correct += 1
            elif not any(results):
                all_wrong += 1
            else:
                mixed += 1
        overlap_data[subject] = (all_correct, mixed, all_wrong, num_q)

    # ============================================================================
    # 8. Question Overlap Stacked Bar — how many questions do all models agree on?
    # ============================================================================
    print("Creating question overlap graph...")

    subj_labels  = list(overlap_data.keys())
    all_correct_pct = [overlap_data[s][0] / overlap_data[s][3] * 100 for s in subj_labels]
    mixed_pct       = [overlap_data[s][1] / overlap_data[s][3] * 100 for s in subj_labels]
    all_wrong_pct   = [overlap_data[s][2] / overlap_data[s][3] * 100 for s in subj_labels]

    fig, ax = plt.subplots(figsize=(14, 7))
    x = np.arange(len(subj_labels))
    b1 = ax.bar(x, all_correct_pct, label='All models correct', color='#4CAF50', alpha=0.85)
    b2 = ax.bar(x, mixed_pct,       bottom=all_correct_pct,     label='Mixed results',       color='#FFC107', alpha=0.85)
    b3 = ax.bar(x, all_wrong_pct,
                bottom=[a + b for a, b in zip(all_correct_pct, mixed_pct)],
                label='All models wrong', color='#F44336', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(subj_labels, rotation=45, ha='right')
    ax.set_ylabel('Percentage of Questions (%)', fontsize=12)
    ax.set_title('Question-Level Agreement Across All Models per Subject', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('mmlu_analysis_plots/8_question_overlap.png', dpi=300, bbox_inches='tight')
    print("  Saved: mmlu_analysis_plots/8_question_overlap.png")

    # ============================================================================
    # 9. Pairwise Model Agreement Heatmap
    # ============================================================================
    print("Creating pairwise agreement heatmap...")

    n = len(models)
    agreement_matrix = np.zeros((n, n))

    for i, res_i in enumerate(data['results']):
        for j, res_j in enumerate(data['results']):
            total_q = 0
            agree_q = 0
            for subj in subjects:
                sr_i = next((s for s in res_i['subject_results'] if s['subject'] == subj), None)
                sr_j = next((s for s in res_j['subject_results'] if s['subject'] == subj), None)
                if sr_i is None or sr_j is None:
                    continue
                for qi, qj in zip(sr_i['question_results'], sr_j['question_results']):
                    if qi['predicted_answer'] == qj['predicted_answer']:
                        agree_q += 1
                    total_q += 1
            agreement_matrix[i, j] = agree_q / total_q * 100 if total_q else 0

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(agreement_matrix, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.set_yticklabels(models)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Answer Agreement (%)', rotation=270, labelpad=20)

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{agreement_matrix[i, j]:.1f}%',
                    ha='center', va='center', fontsize=10,
                    color='white' if agreement_matrix[i, j] > 60 else 'black')

    ax.set_title('Pairwise Model Answer Agreement\n(% questions with same answer choice)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('mmlu_analysis_plots/9_pairwise_agreement.png', dpi=300, bbox_inches='tight')
    print("  Saved: mmlu_analysis_plots/9_pairwise_agreement.png")

# ============================================================================
# Summary Statistics
# ============================================================================
print("\n" + "="*70)
print("ANALYSIS SUMMARY")
print("="*70)

subject_avg_accs = [(s, np.mean(subject_data[s])) for s in subjects]
subject_avg_accs.sort(key=lambda x: x[1], reverse=True)

if has_timing:
    avg_real_times = [r['timing']['avg_real_time_per_question'] for r in data['results']]
    print(f"\nBest Overall Model:  {models[np.argmax(overall_accuracies)]} ({max(overall_accuracies):.2f}%)")
    print(f"Fastest Model:       {models[np.argmin(avg_real_times)]} ({min(avg_real_times):.3f}s per question)")

print("\nSubject Difficulty (average across all models):")
print("  Easiest:")
for subject, acc in subject_avg_accs[:3]:
    print(f"    {subject}: {acc:.2f}%")
print("  Hardest:")
for subject, acc in subject_avg_accs[-3:]:
    print(f"    {subject}: {acc:.2f}%")

print("\nModel Strengths (subjects where each model ranks #1):")
for i, model in enumerate(models):
    best = [s for s in subjects if subject_data[s] and subject_data[s][i] == max(subject_data[s])]
    if best:
        print(f"  {model}: {', '.join(best)}")

if has_questions:
    print("\nQuestion Overlap Summary:")
    total_all_wrong = sum(overlap_data[s][2] for s in overlap_data)
    total_all_q = sum(overlap_data[s][3] for s in overlap_data)
    print(f"  Questions ALL models got wrong: {total_all_wrong}/{total_all_q} "
          f"({total_all_wrong/total_all_q*100:.1f}%)")
    total_all_right = sum(overlap_data[s][0] for s in overlap_data)
    print(f"  Questions ALL models got right: {total_all_right}/{total_all_q} "
          f"({total_all_right/total_all_q*100:.1f}%)")

print("\n" + "="*70)
print("✓ All graphs saved to mmlu_analysis_plots/")
print("="*70)

Overwriting analyze_mmlu_results.py


# Run Analysis Script to Visualize Multimodel MMLU Results

In [22]:
!python analyze_mmlu_results.py multimodel_mmlu_results_a100_20260302_000534.json

Creating accuracy comparison graph...
  Saved: mmlu_analysis_plots/1_overall_accuracy.png
Creating subject comparison graph...
  Saved: mmlu_analysis_plots/2_subject_comparison.png
Creating timing comparison graphs...
  Saved: mmlu_analysis_plots/3_timing_comparison.png
Creating accuracy vs speed graph...
  Saved: mmlu_analysis_plots/4_accuracy_vs_speed.png
Creating performance heatmap...
  Saved: mmlu_analysis_plots/5_performance_heatmap.png
Creating correct/wrong breakdown...
  Saved: mmlu_analysis_plots/6_correct_wrong_breakdown.png
Creating subject rankings...
  Saved: mmlu_analysis_plots/7_subject_rankings.png
Creating question overlap graph...
  Saved: mmlu_analysis_plots/8_question_overlap.png
Creating pairwise agreement heatmap...
  Saved: mmlu_analysis_plots/9_pairwise_agreement.png

ANALYSIS SUMMARY

Best Overall Model:  Qwen2.5-7B-Instruct (65.87%)
Fastest Model:       OLMo-1B-0724-hf (0.019s per question)

Subject Difficulty (average across all models):
  Easiest:
    inter